In [2]:
from elasticsearch import Elasticsearch

In [4]:
es = Elasticsearch(
    "https://localhost:9200",
    basic_auth=("elastic","LQym+efHnUy9DbT-jtD2"),
    ca_certs="C:\\Users\\Lenovo\\Downloads\\elasticsearch-9.4.1-windows-x86_64\\elasticsearch-9.4.1\\config\\certs\\http_ca.crt"
)
es.ping()

False

## Prepare the data

In [5]:
import pandas as pd

df = pd.read_csv("myntra_products_catalog.csv").loc[:499]
df.head()

,ProductID,ProductName,ProductBrand,Gender,Price (INR),NumImages,Description,PrimaryColor
0,10017413,DKNY Unisex Black & Grey Printed Medium Trolle...,DKNY,Unisex,11745,7,"Black and grey printed medium trolley bag, sec...",Black
1,10016283,EthnoVogue Women Beige & Grey Made to Measure ...,EthnoVogue,Women,5810,7,Beige & Grey made to measure kurta with churid...,Beige
2,10009781,SPYKAR Women Pink Alexa Super Skinny Fit High-...,SPYKAR,Women,899,7,Pink coloured wash 5-pocket high-rise cropped ...,Pink
3,10015921,Raymond Men Blue Self-Design Single-Breasted B...,Raymond,Men,5599,5,Blue self-design bandhgala suitBlue self-desig...,Blue
4,10017833,Parx Men Brown & Off-White Slim Fit Printed Ca...,Parx,Men,759,5,"Brown and off-white printed casual shirt, has ...",White


In [6]:
df.isna().value_counts()

ProductID  ProductName  ProductBrand  Gender  Price (INR)  NumImages  Description  PrimaryColor
False      False        False         False   False        False      False        False           468
                                                                                   True             32
Name: count, dtype: int64

In [7]:
df.fillna("None", inplace=True)

,ProductID,ProductName,ProductBrand,Gender,Price (INR),NumImages,Description,PrimaryColor
0,10017413,DKNY Unisex Black & Grey Printed Medium Trolle...,DKNY,Unisex,11745,7,"Black and grey printed medium trolley bag, sec...",Black
1,10016283,EthnoVogue Women Beige & Grey Made to Measure ...,EthnoVogue,Women,5810,7,Beige & Grey made to measure kurta with churid...,Beige
2,10009781,SPYKAR Women Pink Alexa Super Skinny Fit High-...,SPYKAR,Women,899,7,Pink coloured wash 5-pocket high-rise cropped ...,Pink
3,10015921,Raymond Men Blue Self-Design Single-Breasted B...,Raymond,Men,5599,5,Blue self-design bandhgala suitBlue self-desig...,Blue
4,10017833,Parx Men Brown & Off-White Slim Fit Printed Ca...,Parx,Men,759,5,"Brown and off-white printed casual shirt, has ...",White
...,...,...,...,...,...,...,...,...
495,10018075,Puma Men Blue Sneakers,Puma,Men,1749,5,"A pair of round-toe blue sneakers, has regular...",Blue
496,10009687,SPYKAR Women Blue & White Striped Adora Skinny...,SPYKAR,Women,1034,7,Blue and White striped 5-pocket mid-rise jeans...,Blue
497,10017425,DKNY Unisex Black & Grey Printed Large Trolley...,DKNY,Unisex,13275,7,"Black and grey printed large trolley bag, secu...",Black
498,10017615,Parx Men Pink Slim Fit Solid Casual Shirt,Parx,Men,612,5,"Pink solid casual shirt, has a spread collar, ...",Pink


## Convert the relevant field to Vector using BERT model

In [8]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-mpnet-base-v2')

d:\Data Science\semantic-search-elastic-search-and-BERT-vector-embedding\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Data Science\semantic-search-elastic-search-and-BERT-vector-embedding\env\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you e

In [9]:
df["DescriptionVector"] = df["Description"].apply(lambda x: model.encode(x))

In [10]:
df.head()

,ProductID,ProductName,ProductBrand,Gender,Price (INR),NumImages,Description,PrimaryColor,DescriptionVector
0,10017413,DKNY Unisex Black & Grey Printed Medium Trolle...,DKNY,Unisex,11745,7,"Black and grey printed medium trolley bag, sec...",Black,"[0.027645878, -0.002634183, -0.0035884008, 0.0..."
1,10016283,EthnoVogue Women Beige & Grey Made to Measure ...,EthnoVogue,Women,5810,7,Beige & Grey made to measure kurta with churid...,Beige,"[-0.024660693, -0.028755356, -0.020332504, 0.0..."
2,10009781,SPYKAR Women Pink Alexa Super Skinny Fit High-...,SPYKAR,Women,899,7,Pink coloured wash 5-pocket high-rise cropped ...,Pink,"[-0.04694326, 0.08182797, 0.048335165, -0.0001..."
3,10015921,Raymond Men Blue Self-Design Single-Breasted B...,Raymond,Men,5599,5,Blue self-design bandhgala suitBlue self-desig...,Blue,"[-0.015098731, -0.010285397, 0.009487311, -0.0..."
4,10017833,Parx Men Brown & Off-White Slim Fit Printed Ca...,Parx,Men,759,5,"Brown and off-white printed casual shirt, has ...",White,"[-0.017746568, 0.0062096356, 0.021813974, 0.02..."


In [11]:
es.ping()

False

## Create new index in ElasticSearch!

In [13]:
from elasticsearch import Elasticsearch

es = Elasticsearch(
"https://localhost:9200",
basic_auth=("elastic", "AUHVn9oi7sAmpO6vM_zB"),
verify_certs=False
)

print(es.info())

from indexMapping import indexMapping

es.indices.create(
index="all_products",
mappings=indexMapping
)


d:\Data Science\semantic-search-elastic-search-and-BERT-vector-embedding\env\Lib\site-packages\elasticsearch\_sync\client\__init__.py:326: SecurityWarning: Connecting to 'https://localhost:9200' using TLS with verify_certs=False is insecure
  _transport = transport_class(
d:\Data Science\semantic-search-elastic-search-and-BERT-vector-embedding\env\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
d:\Data Science\semantic-search-elastic-search-and-BERT-vector-embedding\env\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


{'name': 'DESKTOP-K1HPRJM', 'cluster_name': 'elasticsearch', 'cluster_uuid': 'rXGPLVryStaHzX_VQHvXaQ', 'version': {'number': '9.4.1', 'build_flavor': 'default', 'build_type': 'zip', 'build_hash': '3c7c6027c5769d860d87448e2749f4c550a239da', 'build_date': '2026-05-08T10:08:29.383338563Z', 'build_snapshot': False, 'lucene_version': '10.4.0', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'}


ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'all_products'})

## Ingest the data into index

In [14]:
record_list = df.to_dict("records")

In [15]:
for record in record_list:
    try:
        es.index(index="all_products", document=record, id=record["ProductID"])
    except Exception as e:
        print(e)

d:\Data Science\semantic-search-elastic-search-and-BERT-vector-embedding\env\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
d:\Data Science\semantic-search-elastic-search-and-BERT-vector-embedding\env\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
d:\Data Science\semantic-search-elastic-search-and-BERT-vector-embedding\env\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://url

In [16]:
es.count(index="all_products")

d:\Data Science\semantic-search-elastic-search-and-BERT-vector-embedding\env\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


ObjectApiResponse({'count': 441, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}})

## Search the data

In [ ]:
input_keyword = "fruits"

vector_of_input_keyword = model.encode(input_keyword).tolist()

res = es.search(
index="all_products",
knn={
"field": "DescriptionVector",
"query_vector": vector_of_input_keyword,
"k": 2,
"num_candidates": 500
},
source=["ProductName", "Description"]
)

for hit in res["hits"]["hits"]:
    print("Product Name:", hit["_source"]["ProductName"])
    print("Description:", hit["_source"]["Description"])
    print("Score:", hit["_score"])
    print("-" * 40)


d:\Data Science\semantic-search-elastic-search-and-BERT-vector-embedding\env\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


[{'_index': 'all_products', '_id': '10002003', '_score': 0.44539022, '_source': {'ProductName': 'Police Men Imperial Patchouli Eau de Toilette 100 ml', 'Description': 'NotesTop notes: Mandarin orange, spices and bergamotHeart notes: Green and floral notesBase notes: Patchouli and woody notes'}}, {'_index': 'all_products', '_id': '10001527', '_score': 0.44124958, '_source': {'ProductName': 'Bvlgari Splendida Women Rose Eau De Parfum 50 ml', 'Description': 'NotesTop notes: Mandarin Orange and BlackberryHeart notes: Damask RoseBase notes: Patchouli, Sandalwood, Vetiver and Musk'}}]
